# PYTORCH IMPLEMENTATION
Method 1 (Educational)

# Converting patches to embeddings using different methods

In [1]:
import torch
import torch.nn as nn

image = torch.randn(1, 3, 224, 224)

patch_size = 16

unfold = nn.Unfold(
    kernel_size=patch_size,
    stride=patch_size
)

patches = unfold(image)

print(patches.shape)

torch.Size([1, 768, 196])


## REARRANGE

In [ ]:
patches = patches.transpose(1,2)

print(patches.shape)

# Batch
# Tokens
# Features

torch.Size([1, 196, 768])


## LINEAR PROJECTION

In [3]:
projection = nn.Linear(
    768,
    768
)

embeddings = projection(patches)

print(embeddings.shape)

torch.Size([1, 196, 768])


# METHOD 2 (REAL ViT IMPLEMENTATION)

In [4]:
import torch
import torch.nn as nn

patch_embed = nn.Conv2d(
    in_channels=3,
    out_channels=768,
    kernel_size=16,
    stride=16
)

image = torch.randn(1,3,224,224)

x = patch_embed(image)

print(x.shape)

torch.Size([1, 768, 14, 14])


## FINAL RESHAPING

In [ ]:
x = x.flatten(2)
x = x.transpose(1,2)

print(x.shape)


# WHAT HAVE WE ACHIEVED?

# Starting from:

# 224×224×3

# we obtained:

# (196,768)

# tokens.

# Now the image looks like a sentence:

# Token1
# Token2
# Token3
# ...
# Token196

# And because it now looks like a sequence, we can feed it into a Transformer Encoder.

torch.Size([1, 196, 768])


# INTUITION BOX

CNN thinks:

Image

 ↓
Feature Maps

 ↓
Feature Maps

 ↓
Feature Maps


ViT thinks:

Image

 ↓
Patch Tokens

 ↓
Transformer

 ↓
Patch Tokens


The Transformer never sees the original image.

It only sees token embeddings.

## We currently have:

(196,768)

tokens.

But there are two major problems:

Problem 1

Which token represents the entire image?

Need:

CLS Token
Problem 2

Transformer doesn't know:

Top Left
Bottom Right
Center

Need:

Positional Embeddings

# CHAPTER 4 : CLS TOKEN

At this point we have:

(196,768)

meaning:

196 patch tokens

768-dimensional embedding for each token

## HOW DOES THE MODEL CLASSIFY THE IMAGE?

Suppose image:

Dog

After patchification:

Patch1
Patch2
Patch3
...
Patch196

Transformer processes all patches.

But finally we need:

Dog

as output.

Which token should be used?

FIRST NAIVE IDEA

Use all tokens.

196 tokens

 ↓

Classifier

 ↓
 
Prediction

Possible.

But computationally expensive.

And doesn't provide a dedicated representation of the whole image.

# PYTORCH IMPLEMENTATION

Create learnable CLS token.

In [ ]:
import torch
import torch.nn as nn

cls_token = nn.Parameter( # this means that the tensor which is made is trainable
    torch.randn(1,1,768)
)

print(cls_token.shape)

torch.Size([1, 1, 768])


In [7]:
batch_size = 32
cls_tokens = cls_token.expand(
    batch_size,
    -1,
    -1
)

print(cls_tokens.shape)

torch.Size([32, 1, 768])


## CONCATENATE

In [11]:
patch_embeddings = torch.randn(32, 196, 768)
cls_tokens = torch.randn(32, 1, 768)

x = torch.cat((cls_tokens, patch_embeddings), dim=1)

print(x.shape)

torch.Size([32, 197, 768])


# CHAPTER 5 : POSITIONAL EMBEDDINGS

Now comes the second major problem.

THE PROBLEM

Suppose we shuffle tokens.

P1 P2 P3 P4

↓

P3 P1 P4 P2

Transformer doesn't know anything changed.

Why?

Because attention is permutation invariant.

What does permutation invariant mean?
Definition

A permutation invariant model treats reordered inputs identically unless positional information is explicitly provided.

IMAGE EXAMPLE

Suppose:

Cat at top


Dog at bottom


and



Dog at top


Cat at bottom


These are different images.


But without positions:


Transformer sees only tokens


No coordinates.


No location.


No ordering.


SOLUTION


Add positional information.


Just like NLP.


EXAMPLE

Patch embedding:

[0.2, 0.5, 0.8]

Position embedding:

[0.1, 0.3, 0.4]

Result:

[0.3, 0.8, 1.2]

LEARNED POSITION EMBEDDINGS

ViT uses:

Learnable Positional Embeddings

instead of sinusoidal embeddings.

Each position has its own learnable vector.

# PYTORCH IMPLEMENTATION

In [12]:
pos_embedding = nn.Parameter(
    torch.randn(1,197,768)
)

print(pos_embedding.shape)

torch.Size([1, 197, 768])


## ADD TO INPUT

Current input:

(32,197,768)

Add:

(1,197,768)

PyTorch broadcasts automatically.

x = x + pos_embedding

Output:

(32,197,768)

# CHAPTER 6 : COMPLETE VISION TRANSFORMER ARCHITECTURE

Researchers realized:

Transformer Encoder

++

Image Patches

=

Vision Transformer

ONE TRANSFORMER ENCODER BLOCK

Input

 ↓
 
LayerNorm

 ↓
 
Multi-Head Attention

 ↓
 
Residual Connection

 ↓
 
LayerNorm

 ↓
 
MLP

 ↓
 
Residual Connection

# STEP 1 : LAYER NORMALIZATION

Input:

(32,197,768)

Apply LayerNorm.

Output:

(32,197,768)

Shape unchanged.

Purpose:

Stable training

# STEP 2 : MULTI-HEAD ATTENTION

Input:

(32,197,768)

Generate:

Queries
Keys
Values

for every token.

Remember:

197 tokens

=
1 CLS
+
196 Patches
IMPORTANT DIFFERENCE FROM CNN

CNN:

Pixel interacts with nearby pixels

ViT:

Every patch can interact
with every other patch

in a single layer.

ATTENTION MATRIX

For:

197 tokens

Attention matrix size:

197×197

Each token attends to every token.

Visual:

      CLS P1 P2 P3
CLS    ✓  ✓  ✓  ✓

P1     ✓  ✓  ✓  ✓

P2     ✓  ✓  ✓  ✓

P3     ✓  ✓  ✓  ✓

Every token communicates with every other token.

COMPUTATIONAL COST

Attention complexity:

O(N
2
)

where:

N=Number of Tokens

For ViT:

N=197

manageable.

But for larger images:

More patches
↓
Larger attention matrix
↓
Higher memory

This later motivated Swin Transformer.

MULTI-HEAD ATTENTION OUTPUT

Input:

(32,197,768)

Output:

(32,197,768)

Shape unchanged.

But representations become richer.

# STEP 3 : FIRST RESIDUAL CONNECTION
Formula:

x=x+Attention(x)

You already studied this.

Purpose:

Better gradient flow
Prevent information loss

Output:

(32,197,768)

# STEP 4 : SECOND LAYERNORM

Again:

(32,197,768)

Shape unchanged.

# STEP 5 : MLP BLOCK

This is often overlooked in interviews.

Many students focus only on attention.

But MLP is extremely important.

Structure

Typically:

768
 ↓
3072
 ↓
768

Visual:

Linear
 ↓
GELU
 ↓
Linear
WHY EXPAND TO 3072?

Embedding dimension:

768

Hidden dimension:

3072

Expansion factor:

4×

This gives higher representational capacity.

SECOND RESIDUAL CONNECTION
Formula:

x=x+MLP(x)

Output:

(32,197,768)